# Apply Location Curations

Lays location curations over `openalex.works.locations_mapped` in the view `openalex.works.locations_mapped_curated`. Runs after `Locations_Mapped` and `Sync_Location_Curations`, and before `Works_Base`; CreateWorksBase and the locations ES sync read the view. Moved out of CreateLocationsMapped (matching only), oxjob #745.

## Curations are a view, not a write (oxjob #1170, 2026-10-23)

As for work-author curations (`ApplyWorkAuthorCurations`, #1438), curations are laid over the table instead of written into it: CreateLocationsMapped owns `locations_mapped` and rebuilds it every night, and the view recomputes the served values from tonight's table and tonight's curations. A plain view, not a materialized one: the curations are a few thousand rows, broadcast to every worker, and timed at no measurable cost over reading the table alone (oxjob #1170 README, Log 2026-10-23).

Until 2026-10-23 this notebook MERGEd the legacy store `openalex.curations.approved_curations` into the table and inserted curation-created locations. Those field curations now live in users-api (migrated 2026-10-23), and the curation-created locations were dropped (oxjob #1170 README, Retiring the legacy location curations).

## Semantics (oxjob #1170 README)

- Curations come from users-api `POST /curations` (`openalex.works.location_curations`, synced by SyncLocationCurations). Fields: `is_oa`, `license`, `pdf_url`, `version`, `landing_page_url`, and since 2026-11-19 `source_id` (oxjob #2173).
- The latest curation per (location, field) wins (`created DESC, curation_id DESC`). A removal (the licence, or since 2026-11-10 the PDF link: Kyle, oxjob #1170; since 2026-11-22 the landing page, ZD 3533) clears the field; its stored value is `'null'`, so read the action. Every field that can be removed reads its value only on `replace`. A landing-page removal also drops the record's `html` entries from `urls` (2026-11-23): CreateWorksBase falls back to the first of them when `landing_page_url` is NULL.
- A curation names a location by the API's id, `<native_id_namespace>:<native_id>`, and applies to every row with that id (the same DOI from Crossref and from DataCite is two rows).
- `source_id` (oxjob #2173) moves a location to the venue that really holds it, e.g. MAG records of IEICE's 'SAT' technical reports filed under the SAT satisfiability conference. Its value is the source's URL (`https://openalex.org/S123`); a removal leaves the location with no source. `is_oa_source` follows the curated source, so every reader sees the new venue whole. A curation naming a source that is no longer live is skipped (the location keeps its matched source) rather than serving a dangling id.
- `curated_fields` lists the fields a curation set on the row: CreateWorksBase uses those as-is and applies the journal rules (`source_is_oa`, `doaj_license`) only to the rest.

## The view

Every `locations_mapped` column, the six curatable fields (and `is_oa_source`, which follows `source_id`) replaced by their curated values where a curation set them, `urls` without its html entries where a curation removed the landing page, plus `curated_fields`. The curations collapse to one row per location before the join, so the view has exactly one row per `locations_mapped` row.

In [ ]:
%sql
CREATE OR REPLACE VIEW openalex.works.locations_mapped_curated
COMMENT 'locations_mapped with the current location curations laid on top (ApplyLocationCurations, oxjob #1170). Read this, not locations_mapped, for the served values of a location.'
AS
WITH latest AS (
  SELECT location_id, property, action, value
  FROM openalex.works.location_curations
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY location_id, property
    ORDER BY created DESC, curation_id DESC
  ) = 1
),
curated AS (
  SELECT
    split(location_id, ':')[0]                                              AS native_id_namespace,
    substring(location_id, locate(':', location_id) + 1)                    AS native_id,
    MAX(CASE WHEN property = 'is_oa' THEN value = 'true' END)               AS is_oa,
    MAX(CASE WHEN property = 'license' AND action = 'replace' THEN value END) AS license,
    MAX(CASE WHEN property = 'pdf_url' AND action = 'replace' THEN value END) AS pdf_url,
    MAX(CASE WHEN property = 'version' THEN value END)                      AS version,
    MAX(CASE WHEN property = 'landing_page_url' AND action = 'replace' THEN value END) AS landing_page_url,
    MAX(CASE WHEN property = 'source_id' AND action = 'replace'
             THEN CAST(regexp_extract(value, 'S([0-9]+)$', 1) AS BIGINT) END)       AS source_id,
    collect_set(property)                                                   AS curated_fields
  FROM latest
  GROUP BY location_id
),
-- oxjob #2173: the live sources curations name. A curation naming a source that is gone (merged, deleted) finds no row
-- here and is skipped, so the location keeps its matched source instead of a dangling id.
curated_sources AS (
  SELECT id, MAX(is_oa) AS is_oa
  FROM openalex.sources.sources
  WHERE id IN (SELECT source_id FROM curated WHERE source_id IS NOT NULL)
  GROUP BY id
),
curated_live AS (
  SELECT c.*, cs.is_oa AS source_is_oa,
         COALESCE(array_contains(c.curated_fields, 'source_id') AND (c.source_id IS NULL OR cs.id IS NOT NULL), FALSE)
           AS source_curated  -- a removal (source_id NULL), or a live source
  FROM curated c
  LEFT JOIN curated_sources cs ON cs.id = c.source_id
)
SELECT
  lm.* EXCEPT (is_oa, license, pdf_url, version, landing_page_url, source_id, is_oa_source, urls),
  IF(array_contains(c.curated_fields, 'is_oa'), c.is_oa, lm.is_oa)                                  AS is_oa,
  IF(array_contains(c.curated_fields, 'license'), c.license, lm.license)                            AS license,
  IF(array_contains(c.curated_fields, 'pdf_url'), c.pdf_url, lm.pdf_url)                            AS pdf_url,
  IF(array_contains(c.curated_fields, 'version'), c.version, lm.version)                            AS version,
  IF(array_contains(c.curated_fields, 'landing_page_url'), c.landing_page_url, lm.landing_page_url) AS landing_page_url,
  -- oxjob #1170 (ZD 3533): a removed landing page also leaves the record's html urls. CreateWorksBase reads the first html url
  -- as the landing page when landing_page_url is NULL, and the locations index serves `urls`, so the removed link came back
  -- (night 2026-11-23, mag:1714723801). Only a removal touches urls; a replace already wins over the fallback.
  -- Revert: drop this IF and `urls` from the EXCEPT list.
  IF(COALESCE(array_contains(c.curated_fields, 'landing_page_url') AND c.landing_page_url IS NULL, FALSE),
     filter(lm.urls, u -> u.content_type IS DISTINCT FROM 'html'), lm.urls)                         AS urls,
  IF(COALESCE(c.source_curated, FALSE), c.source_id, lm.source_id)                                  AS source_id,
  IF(COALESCE(c.source_curated, FALSE), c.source_is_oa, lm.is_oa_source)                           AS is_oa_source,
  COALESCE(c.curated_fields, CAST(array() AS ARRAY<STRING>))                                        AS curated_fields
FROM openalex.works.locations_mapped lm
LEFT JOIN curated_live c
  ON c.native_id_namespace = lm.native_id_namespace AND c.native_id = lm.native_id

## Verify

Rows where the view differs from the table are the rows curations change tonight. Only rows with `curated_fields` can differ, so both cells look at those only.

In [ ]:
%sql
SELECT
  (SELECT COUNT(*) FROM openalex.works.location_curations)       AS location_curations_total,
  COUNT(*)                                                        AS curated_rows,
  COUNT_IF(NOT (v.is_oa <=> lm.is_oa))                            AS is_oa_changed,
  COUNT_IF(NOT (v.license <=> lm.license))                        AS license_changed,
  COUNT_IF(NOT (v.pdf_url <=> lm.pdf_url))                        AS pdf_url_changed,
  COUNT_IF(NOT (v.version <=> lm.version))                        AS version_changed,
  COUNT_IF(NOT (v.landing_page_url <=> lm.landing_page_url))      AS landing_page_url_changed,
  COUNT_IF(NOT (v.source_id <=> lm.source_id))                    AS source_id_changed
FROM openalex.works.locations_mapped_curated v
JOIN openalex.works.locations_mapped lm
  ON  lm.native_id_namespace = v.native_id_namespace AND lm.native_id = v.native_id
  AND lm.provenance = v.provenance AND lm.work_id <=> v.work_id
WHERE size(v.curated_fields) > 0

In [ ]:
%sql
-- Spot-check: rows the curations change tonight, table value beside served value.
SELECT concat(v.native_id_namespace, ':', v.native_id) AS location_id, v.provenance, v.work_id, v.curated_fields,
       lm.is_oa AS table_is_oa, v.is_oa AS served_is_oa,
       lm.license AS table_license, v.license AS served_license,
       lm.pdf_url AS table_pdf_url, v.pdf_url AS served_pdf_url,
       lm.source_id AS table_source_id, v.source_id AS served_source_id
FROM openalex.works.locations_mapped_curated v
JOIN openalex.works.locations_mapped lm
  ON  lm.native_id_namespace = v.native_id_namespace AND lm.native_id = v.native_id
  AND lm.provenance = v.provenance AND lm.work_id <=> v.work_id
WHERE size(v.curated_fields) > 0
ORDER BY location_id
LIMIT 100